In [3]:

import json
import hashlib
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import os
import sys
sys.path.append("..")
from utils.balanced_builders_hdf import *
from utils.losses import *
from utils.recon_error import *
from utils.trainer import *
from models.autoencoder_classifier import *
from config import *

In [4]:
cs = build_balanced_cs_loaders_from_h5(
    h5_path=xrd_dataset,
    per_class_cs=105000,   # adjust as you want
    batch_size=256,
    val_split=0.1,
    test_split=0.1,
    seed=42,
    num_workers=4,
)

print(cs["counts"])
print(cs["sizes"])


{'triclinic': 105000, 'monoclinic': 105000, 'orthorhombic': 105000, 'tetragonal': 105000, 'trigonal': 105000, 'hexagonal': 105000, 'cubic': 105000}
{'train': 588000, 'val': 73500, 'test': 73500}


In [ ]:

# =====================================================================
# Training loop 
# =====================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"

model = DeepConvAutoencoderClassifier(
    input_length=cs["input_len"],
    latent_dim=64,         
    cls_dim=128,            
    num_classes=cs["num_classes"],
    use_projection_head=True
).to(device)

def make_optimizer(model, lr=1e-3, wd=1e-4):
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=wd
    )

optimizer = make_optimizer(model)

# ===== Balanced: good reconstruction + stronger classifier, mild contrastive =====
params = {
    # Reconstruction vs classification balance
    "recon_mul":   1.5,    
    "alpha_cls":   7.0,    
    "beta_smooth": 0.02,   

    # Augmentation / contrastive
    "noise_std":   0.02,   
    "lambda_ntx":  0.00,   
    "lambda_sup":  0.2,    
}


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Disable augmentation/contrastive completely
params["noise_std"]  = 0.0
params["lambda_ntx"] = 0.0
params["lambda_sup"] = 0.2

num_epochs = 200
best_val_acc = 0.0

save_path_clf   = CS_Cls
save_path_recon = CS_RECO
for epoch in range(1, num_epochs + 1):
    logs = {"loss": 0.0, "recon": 0.0, "clf": 0.0, "smooth": 0.0, "ntx": 0.0, "supcon": 0.0}
    steps = 0

    # -------------------------
    # Training
    # -------------------------
    for x, y in cs["train_loader"]:
        out = train_step(
            model=model,
            optimizer=optimizer,
            x=x,
            y=y,
            epoch=epoch,
            params=params,
            device=device,
            use_augmentation=False,   # <-- NO augmentation
        )

        for k in logs:
            logs[k] += out[k]
        steps += 1

    for k in logs:
        logs[k] /= max(1, steps)

    # -------------------------
    # Validation
    # -------------------------
    val = val_step(model, cs["val_loader"], device)
    val_acc   = val["val_acc"]
    val_recon = val["val_recon"]

    # -------------------------
    # Save best classifier
    # -------------------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path_clf)
        torch.save(model.state_dict(), save_path_recon)
        print(f"✓ Saved BEST CLASSIFIER at epoch {epoch} | ValAcc = {best_val_acc:.4f}")

    # -------------------------
    # Logging
    # -------------------------
    print(
        f"Epoch {epoch:03d} | "
        f"Loss {logs['loss']:.4f} | Recon {logs['recon']:.4f} | "
        f"Clf {logs['clf']:.4f} | Smooth {logs['smooth']:.4f} | "
        f"NTX {logs['ntx']:.4f} | SupCon {logs['supcon']:.4f} || "
        f"ValAcc {val_acc:.4f} | ValRecon {val_recon:.4f}"
    )

# Save final model after full training
torch.save(model.state_dict(), save_path_recon)
print(f"Saved FINAL model after full {num_epochs} epochs → {save_path_recon}")

✓ Saved BEST CLASSIFIER at epoch 1 | ValAcc = 0.8647
Epoch 001 | Loss 4.8348 | Recon 0.0109 | Clf 0.6883 | Smooth 0.0034 | NTX 0.0000 | SupCon 6.4923 || ValAcc 0.8647 | ValRecon 0.0064
✓ Saved BEST CLASSIFIER at epoch 2 | ValAcc = 0.9441
Epoch 002 | Loss 1.6291 | Recon 0.0060 | Clf 0.2314 | Smooth 0.0034 | NTX 0.0000 | SupCon 6.3334 || ValAcc 0.9441 | ValRecon 0.0056
Epoch 003 | Loss 0.7602 | Recon 0.0054 | Clf 0.1074 | Smooth 0.0034 | NTX 0.0000 | SupCon 6.2794 || ValAcc 0.8933 | ValRecon 0.0054
✓ Saved BEST CLASSIFIER at epoch 4 | ValAcc = 0.9684
Epoch 004 | Loss 0.5091 | Recon 0.0049 | Clf 0.0717 | Smooth 0.0034 | NTX 0.0000 | SupCon 6.2598 || ValAcc 0.9684 | ValRecon 0.0049
Epoch 005 | Loss 0.3970 | Recon 0.0045 | Clf 0.0557 | Smooth 0.0035 | NTX 0.0000 | SupCon 6.2416 || ValAcc 0.9297 | ValRecon 0.0050
✓ Saved BEST CLASSIFIER at epoch 6 | ValAcc = 0.9700
Epoch 006 | Loss 0.3244 | Recon 0.0043 | Clf 0.0454 | Smooth 0.0035 | NTX 0.0000 | SupCon 6.2264 || ValAcc 0.9700 | ValRecon 0.0